# TensorFlow Probability (TFP): A Comprehensive Tutorial
## Using the JAX Substrate

> **Audience**: Developers with strong probability/mathematics background who need to become expert TFP practitioners.
> **Goal**: Provide both practical TFP expertise and genuine understanding of design philosophy and implementation.

### Key References
- [TFP GitHub Repository](https://github.com/tensorflow/probability)
- [TFP Distributions Paper (Dillon et al., 2017)](https://arxiv.org/abs/1711.10604)
- [TFP API Documentation](https://www.tensorflow.org/probability/api_docs/python/tfp)
- [JAX Documentation](https://jax.readthedocs.io/)

### Table of Contents
1. [Introduction](#1-introduction)
2. [Distributions](#2-distributions)
3. [Bijectors](#3-bijectors)
4. [Joint Distributions](#4-joint-distributions)
5. [Conditional Gaussian Example](#5-conditional-gaussian-example)
6. [MCMC](#6-mcmc)

In [ ]:
# Installation (one of the following):
#   pip install tensorflow-probability jax jaxlib matplotlib numpy scipy
#   pip install tensorflow-probability[jax]   # avoids pulling in full TF backend

# Note: this may lead to some issues due to version mismatches between tfp and jax.
#       Solutions include: 
# (1) Downgrade JAX below version 0.7
#       pip install "jax<0.7.0" "jaxlib<0.7.0" "tensorflow-probability[jax]"
# (2) Install tfp-nightly, which has up-to-date experimental (but perhaps unstable) features


import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Import TFP with the JAX substrate ────────────────────────────────────────
# The "substrates" mechanism is a code transformation applied at import time
# that replaces every tf.* call in TFP with the equivalent jax.* call.
# Source: tensorflow_probability/substrates/jax/__init__.py
from tensorflow_probability.substrates import jax as tfp

tfd = tfp.distributions          # Probability distributions
tfb = tfp.bijectors              # Invertible transformations
tfm = tfp.mcmc                   # MCMC transition kernels
tfl = tfp.experimental.linalg    # Linear algebra utilities (optional)

print(f'TFP version : {tfp.__version__}')
print(f'JAX version : {jax.__version__}')
print(f'JAX backend : {jax.default_backend()}')
print(f'JAX devices : {jax.devices()}')

# JAX requires *explicit* PRNG keys.  There is no global random state.
# This is a core part of JAX's functional programming model.
key = jax.random.PRNGKey(42)
print(f'\nSetup complete.')

TFP version : 0.25.0
JAX version : 0.6.2
JAX backend : cpu
JAX devices : [CpuDevice(id=0)]

Setup complete.


---
# 1. Introduction

## 1.1 Brief History

TensorFlow Probability originated at Google Brain and was first described publicly
in the preprint *"TensorFlow Distributions"* (Dillon et al., 2017,
[arXiv:1711.10604](https://arxiv.org/abs/1711.10604)).
Key milestones:

| Year | Milestone |
|------|-----------|
| 2017 | Initial release as *TensorFlow Distributions* — distributions and bijectors |
| 2018 | Renamed to *TensorFlow Probability*; adds MCMC, VI, GPs, STS |
| 2019 | Joint distribution types (`JointDistributionSequential`, etc.) |
| 2020 | **JAX and NumPy substrates** — pure JAX execution with no TF dependency |
| 2021+ | `JointDistributionNamedAutoBatched`, Pathwise gradient estimators, improved GPs |

## 1.2 Design Philosophy

The core insight of TFP, stated in the original paper, is:

> *"We treat probability distributions as first-class objects with a clean,
> composable API."*

Four design pillars:

1. **Composability** — Small primitives (distributions, bijectors) combine into
   arbitrarily complex models via well-defined interfaces.
2. **Vectorisation-first** — Every operation natively supports batching.
   A single `Distribution` object can simultaneously represent millions of
   parameterically-distinct distributions.
3. **Mathematical rigor** — Careful management of shapes, dtypes, log-space
   computations for numerical stability, and proper Jacobian accounting.
4. **Hardware/backend agnosticism** — The same model code runs on CPU, GPU,
   TPU, and (via substrates) with different array frameworks.

## 1.3 The JAX Substrate

The substrate system lives in
`tensorflow_probability/substrates/` and works by importing the full TFP
package while monkey-patching the `tf` namespace with a thin compatibility
layer that routes `tf.*` calls to `jax.*` or `jnp.*` equivalents.

When you write:
```python
from tensorflow_probability.substrates import jax as tfp
```
you get a version of TFP that:
- Uses `jax.numpy` arrays instead of `tf.Tensor`
- Is fully compatible with `jax.jit`, `jax.grad`, `jax.vmap`, `jax.pmap`
- Follows JAX's **functional purity** contract — no global mutable state
- Threads PRNG keys explicitly (JAX's `PRNGKey` rather than TF's `Stateful` seeds)

## 1.4 Core Abstractions at a Glance

| Abstraction | Base class | Purpose |
|-------------|-----------|---------|
| **Distribution** | `tfd.Distribution` | Represent and manipulate probability laws |
| **Bijector** | `tfb.Bijector` | Smooth, invertible maps; change-of-variables |
| **Joint Distribution** | `tfd.JointDistribution*` | Hierarchical / graphical models |
| **MCMC Kernel** | `tfm.TransitionKernel` | Markov chain Monte Carlo samplers |
| **PSD Kernels** | `tfp.math.psd_kernels.*` | Gaussian process covariance kernels |
| **Variational inference** | `tfp.vi.*` | ELBO optimisation utilities |

---
# 2. Distributions

## 2.1 The `tfd.Distribution` Base Class

Every distribution in TFP inherits from
`tfd.Distribution` (source:
`tensorflow_probability/python/distributions/distribution.py`).
The class defines a uniform API that every concrete distribution must implement:

### Core interface

| Method | Signature | Returns |
|--------|-----------|---------|
| `sample` | `(sample_shape=(), seed=None)` | Tensor of samples |
| `log_prob` | `(x)` | Log density / log PMF |
| `prob` | `(x)` | Density / PMF |
| `mean` | `()` | Expected value |
| `variance` | `()` | Variance |
| `stddev` | `()` | Standard deviation |
| `entropy` | `()` | Shannon entropy |
| `covariance` | `()` | Covariance matrix |
| `kl_divergence` | `(other)` | KL divergence D_KL(self ‖ other) |
| `cdf` | `(x)` | Cumulative distribution function |
| `quantile` | `(p)` | Quantile (inverse CDF) |
| `log_cdf` | `(x)` | Log CDF |
| `survival_function` | `(x)` | 1 − CDF |

Internally, each concrete distribution implements *private* methods such as
`_sample_n`, `_log_prob`, `_mean`, etc.  The public methods add common logic
(shape validation, dtype coercion, argument checking) before delegating.

### Lazy vs. eager shape computation
Two variants of each shape property exist:

```python
dist.batch_shape          # tf.TensorShape (static / symbolic)
dist.batch_shape_tensor() # jnp.ndarray    (dynamic, JIT-compatible)
```

## 2.2 Shape Semantics — The Most Important Concept in TFP

TFP adopts a strict, three-tier shape convention:

```
output shape = sample_shape + batch_shape + event_shape
               ^^^^^^^^^^^^^  ^^^^^^^^^^^  ^^^^^^^^^^^^
               how many draws  how many     shape of one
               (passed to      distinct     outcome
               sample())       distributions
```

NOTE: the `+` here is in the Python tuple sense of "append" so that `(1,2) + (3,4,5) + (6,)` is `(1,2,3,4,5,6)`

### Definitions

**`event_shape`** — The shape of *one* outcome drawn from *one* distribution.
- Scalar distribution (Normal): `event_shape = ()`
- Multivariate distribution in $\mathbb{R}^d$: `event_shape = (d,)`
- Distribution over $m \times n$ matrices: `event_shape = (m, n)`

**`batch_shape`** — Represents a collection of *independent* distributions,
each with its own parameters, that live inside a single `Distribution` object.
Think of it as a vectorised plate in plate notation.

**`sample_shape`** — The number of i.i.d. draws, specified at call time via
`dist.sample(sample_shape, seed=...)`.

### Why this design?

Separating batch and event dimensions allows a single Python object to
represent, say, 1 000 000 distinct Gaussians and evaluate their log-probabilities
simultaneously via a single vectorised JAX call — no Python loop required.

### The `Independent` wrapper

`tfd.Independent(dist, reinterpreted_batch_ndims=k)` *promotes* the last `k`
batch dimensions into event dimensions and sums the log-probabilities over them.
This is the standard way to make a "product distribution" (i.e., a distribution
over vectors whose components are independent).

$$
\log p_{\text{Independent}}(\mathbf{x}) = \sum_{i=1}^{d} \log p_i(x_i)
$$

In [3]:
# ── 2.2  Shape semantics — worked examples ────────────────────────────────────

print('='*60)
print('SCALAR DISTRIBUTIONS')
print('='*60)

SCALAR DISTRIBUTIONS


In [7]:
# Single Normal: batch_shape=(), event_shape=()
x = tfd.Normal(loc=0., scale=1.)
print(f'Normal(0,1):')
print(f'  batch_shape={x.batch_shape}, event_shape={x.event_shape}')
s = x.sample(5, seed=key)
print(f'  sample(5).shape = {s.shape}   # sample_shape=(5,) == (5,) + () + ()')
print(f'  log_prob(samples).shape = {x.log_prob(s).shape}  # (5,)')
print(f'  log_prob(0.).shape = {x.log_prob(0.).shape}  # scalar')
print()

Normal(0,1):
  batch_shape=(), event_shape=()
  sample(5).shape = (5,)   # sample_shape=(5,) == (5,) + () + ()
  log_prob(samples).shape = (5,)  # (5,)
  log_prob(0.).shape = ()  # scalar



Now we consider a batch of independent univariate Gaussians. Note that log density evaluations retain the batch structure; i.e., they do not sum over the batch dimensions.

In [19]:
# Batch of univariate Normals: batch_shape=(4,), event_shape=()
x = tfd.Normal(loc=jnp.array([0., 1., 2., 3.]), scale=1.)
s = x.sample(5, seed=key)

print(f'Normal([0,1,2,3], 1):')
print(f'  batch_shape={x.batch_shape}, event_shape={x.event_shape}')
print(f'  sample(5).shape = {s.shape}  # (n_samp, n_batch)')
print(f'  log_prob(samples).shape = {x.log_prob(s).shape}  # (5,4) == (5,) + (4,) == (n_samp,) + (n_batch,)')
print(f'  log_prob([0,0,0,0]).shape = {x.log_prob(jnp.zeros(4)).shape}  # (n_batch,)')
print()

Normal([0,1,2,3], 1):
  batch_shape=(4,), event_shape=()
  sample(5).shape = (5, 4)  # (n_samp, n_batch)
  log_prob(samples).shape = (5, 4)  # (5,4) == (5,) + (4,) == (n_samp,) + (n_batch,)
  log_prob([0,0,0,0]).shape = (4,)  # (n_batch,)



The class stores some other shape information. TFP differentiates between static shapes and dynamic shapes. The former are tuples and
the latter are traceable arrays. In addition, there is information on the shapes of the parameters of the distribution. 
param_shapes/param_static_shapes are methods that take in a sample shape of they argument. Intuitively, they answer the question:
"If I want to generate a specific output shape, what shape do my parameters need to be?"

In [ ]:
# Various other shape information to be aware of
print(f"event_shape: {x.event_shape}")
print(f"batch_shape: {x.batch_shape}")
print(f"is_scalar_event: {x.is_scalar_event()}")
print(f"is_scalar_batch: {x.is_scalar_batch()}")
print(f"event_shape_tensor: {x.event_shape_tensor()}") # returns array (useful if shape should be traced)
print(f"batch_shape_tensor: {x.batch_shape_tensor()}") # same here
print(f"param_shapes(4): {x.param_shapes(4)}") # takes sample shape as argument
print(f"param_static_shapes(4): {x.param_static_shapes(4)}") # same here


event_shape: ()
batch_shape: (4,)
is_scalar_event: True
is_scalar_batch: False
event_shape_tensor: []
batch_shape_tensor: [4]
param_shapes(4): {'loc': Array(4, dtype=int32), 'scale': Array(4, dtype=int32)}
param_static_shapes(4): {'loc': TensorShape([4]), 'scale': TensorShape([4])}


In [36]:
print('='*60)
print('MULTIVARIATE DISTRIBUTIONS')
print('='*60)

MULTIVARIATE DISTRIBUTIONS


In [41]:
# MVN: batch_shape=(), event_shape=(3,)
x = tfd.MultivariateNormalDiag(loc=jnp.zeros(3), scale_diag=jnp.ones(3))
s = x.sample(5, seed=key)

print(f'MVN(dim=3):')
print(f'  batch_shape={x.batch_shape}, event_shape={x.event_shape} == (n_samp,) + (n_batch,) + (n_event,) == (5,) + () + (3,)')
print(f'  sample(5).shape = {s.shape}  # (5, 3)')
print(f'  log_prob(samples).shape = {x.log_prob(s).shape}  # (5,) == (5,) + () == (n_samp,) + (n_batch,)')
print(f'  log_prob(zeros(3)).shape = {x.log_prob(jnp.zeros(3)).shape}  # () == (n_batch,)')
print()

MVN(dim=3):
  batch_shape=(), event_shape=(3,) == (n_samp,) + (n_batch,) + (n_event,) == (5,) + () + (3,)
  sample(5).shape = (5, 3)  # (5, 3)
  log_prob(samples).shape = (5,)  # (5,) == (5,) + () == (n_samp,) + (n_batch,)
  log_prob(zeros(3)).shape = ()  # () == (n_batch,)



In [42]:
# Batch of MVNs: batch_shape=(2,), event_shape=(3,)
x = tfd.MultivariateNormalDiag(
    loc=jnp.zeros((2, 3)), # (n_batch, n_diag)
    scale_diag=jnp.ones((2, 3)),
)
s = x.sample(5, seed=key)

print(f'Batch of 2 MVN(dim=3):')
print(f'  batch_shape={x.batch_shape}, event_shape={x.event_shape}')
print(f'  sample(5).shape = {s.shape}  # (5, 2, 3)')
print(f'  log_prob(samples).shape = {x.log_prob(s).shape}')
print(f'  log_prob(zeros(2,3)).shape = {x.log_prob(jnp.zeros((2,3))).shape}  # (2,)')
print()

Batch of 2 MVN(dim=3):
  batch_shape=(2,), event_shape=(3,)
  sample(5).shape = (5, 2, 3)  # (5, 2, 3)
  log_prob(samples).shape = (5, 2)
  log_prob(zeros(2,3)).shape = (2,)  # (2,)



Note that the sample shape accepts an integer tuple as argument, so samples can be returned in more complicated shapes.

In [43]:
s = x.sample((1, 2, 3), seed=key)

print(f"sample((1,2,3)) = {s.shape} # (1,2,3,2,3) == (1,2,3) + (2,) + (3,)")

sample((1,2,3)) = (1, 2, 3, 2, 3) # (1,2,3,2,3) == (1,2,3) + (2,) + (3,)


The `Independent` class is useful for converting a batch of independent distributions into a single
distribution with independent marginals; i.e., the batch dimensions are re-interpreted as event
dimensions. Log density evaluations will now sum over these dimensions, instead of returning
one value per dimension. Works by interpreting the rightmost `reinterpreted_batch_ndims` batch 
dimensions as event dimensions.

In [46]:
print('='*60)
print('INDEPENDENT: promoting batch dims to event dims')
print('='*60)

# Normal([0,0,0], 1): batch_shape=(3,), event_shape=()
#   log_prob([0,0,0]) -> shape (3,)  -- THREE separate log-probs
base = tfd.Normal(loc=jnp.zeros(3), scale=1.) # Note: example of broadcasting for distribution parameters here
print(f'Normal([0,0,0], 1):')
print(f'  batch_shape={base.batch_shape}, event_shape={base.event_shape}')
print(f'  log_prob([0,0,0]).shape = {base.log_prob(jnp.zeros(3)).shape}')
print(f'  log_prob value = {base.log_prob(jnp.zeros(3))}')

# Independent(Normal([0,0,0], 1), 1): batch_shape=(), event_shape=(3,)
#   log_prob([0,0,0]) -> shape ()   -- ONE joint log-prob (sum of three)
ind = tfd.Independent(base, reinterpreted_batch_ndims=1)
print(f'Independent(Normal([0,0,0],1), 1):')
print(f'  batch_shape={ind.batch_shape}, event_shape={ind.event_shape}')
print(f'  log_prob([0,0,0]).shape = {ind.log_prob(jnp.zeros(3)).shape}')
print(f'  log_prob value = {ind.log_prob(jnp.zeros(3)):.4f} '
      f'(sum of 3 log-probs)')

INDEPENDENT: promoting batch dims to event dims
Normal([0,0,0], 1):
  batch_shape=(3,), event_shape=()
  log_prob([0,0,0]).shape = (3,)
  log_prob value = [-0.9189385 -0.9189385 -0.9189385]
Independent(Normal([0,0,0],1), 1):
  batch_shape=(), event_shape=(3,)
  log_prob([0,0,0]).shape = ()
  log_prob value = -2.7568 (sum of 3 log-probs)


## 2.3 Multivariate Gaussian in Depth

TFP provides several parameterisations of the multivariate Gaussian,
all sharing the same `event_shape` semantics but differing in how the
covariance is stored internally:

| Class | Parameterisation | When to use |
|-------|-----------------|-------------|
| `MultivariateNormalDiag` | $\boldsymbol{\Sigma} = \mathrm{diag}(\sigma^2_1, \ldots, \sigma^2_d)$ | Independent components |
| `MultivariateNormalFullCovariance` | full $\boldsymbol{\Sigma}$ | General covariance |
| `MultivariateNormalTriL` | lower-triangular Cholesky $\mathbf{L}$ s.t. $\boldsymbol{\Sigma} = \mathbf{L}\mathbf{L}^\top$ | Preferred for optimisation |
| `MultivariateNormalLinearOperator` | `tf.linalg.LinearOperator` | Structured covariance (low-rank, etc.) |

The **Cholesky parameterisation** (`TriL`) is almost always preferred because:
- The Cholesky factor $\mathbf{L}$ is unconstrained (up to positivity of diagonal)
- Computing $\log|\det \boldsymbol{\Sigma}| = 2\sum_i \log L_{ii}$ is $O(d)$
- Solving $\boldsymbol{\Sigma}^{-1}\mathbf{v} = \mathbf{L}^{-\top}\mathbf{L}^{-1}\mathbf{v}$ is $O(d^2)$

### Log-probability of the multivariate Gaussian

$$
\log p(\mathbf{x}) = -\frac{d}{2}\log(2\pi) - \frac{1}{2}\log|\boldsymbol{\Sigma}|
                     - \frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})
$$

In [ ]:
# ── 2.3  Multivariate Gaussian ────────────────────────────────────────────────

d = 4
mu = jnp.array([1., 0., -1., 0.5])

# Build a positive-definite covariance via Sigma = A @ A.T + eps*I
key, k1 = jax.random.split(key)
A = jax.random.normal(k1, (d, d)) / jnp.sqrt(d)
Sigma = A @ A.T + 0.5 * jnp.eye(d)
L = jnp.linalg.cholesky(Sigma)   # lower-triangular Cholesky factor

print('Three parameterisations of the same distribution:')
print('='*60)

# 1. Full covariance
mvn_full = tfd.MultivariateNormalFullCovariance(loc=mu, covariance_matrix=Sigma)

# 2. Cholesky (TriL) — preferred
mvn_tril = tfd.MultivariateNormalTriL(loc=mu, scale_tril=L)

# 3. Diagonal (independent components)
mvn_diag = tfd.MultivariateNormalDiag(loc=mu, scale_diag=jnp.sqrt(jnp.diag(Sigma)))

x0 = jnp.zeros(d)
print(f'  log_prob(0) via FullCov  : {mvn_full.log_prob(x0):.6f}')
print(f'  log_prob(0) via TriL     : {mvn_tril.log_prob(x0):.6f}')
print(f'  (Diag ignores off-diag): {mvn_diag.log_prob(x0):.6f}')
print()

# ── Key distribution statistics ──────────────────────────────────────────────
print(f'mean()       shape: {mvn_tril.mean().shape}')           # (d,)
print(f'covariance() shape: {mvn_tril.covariance().shape}')     # (d, d)
print(f'entropy()    shape: {mvn_tril.entropy().shape}')        # ()
print()

key, subkey = jax.random.split(key)
samples = mvn_tril.sample(10_000, seed=subkey)
print(f'sample(10000) shape: {samples.shape}')                  # (10000, 4)
print(f'Empirical mean ≈ true mean:')
print(f'  empirical: {samples.mean(axis=0).round(3)}')
print(f'  true     : {mu.round(3)}')

# ── Accessing the Cholesky factor ─────────────────────────────────────────────
print(f'\nCholesky factor stored internally:')
print(f'  mvn_tril.scale       : {type(mvn_tril.scale)}')
print(f'  mvn_tril.scale.to_dense() shape: {mvn_tril.scale.to_dense().shape}')
# The .scale attribute is a LinearOperator wrapping L.
# This avoids unnecessary materialisation of Sigma = L @ L.T

## 2.4 Categorical Distribution

The `Categorical` distribution represents a discrete distribution over
$\{0, 1, \ldots, k-1\}$ with probabilities $\mathbf{p} = (p_0, \ldots, p_{k-1})$.

$$
P(X = i) = p_i, \quad \sum_{i=0}^{k-1} p_i = 1
$$

TFP accepts either *logits* or *probs*:
- `logits`: unnormalised log-probabilities $\ell_i$, with $p_i = \text{softmax}(\ell)_i$
- `probs`: a probability vector that sums to 1

Prefer **logits** in practice — they live in unconstrained space and avoid
the numerical issues of normalising near-zero probabilities.

Key properties:
- `event_shape = ()` — outcomes are scalars (integers)
- `batch_shape = logits.shape[:-1]` — all but the last dimension

In [ ]:
# ── 2.4  Categorical distribution ─────────────────────────────────────────────

# Single Categorical over 5 classes
logits = jnp.array([2., 1., 0., -1., -2.])
cat = tfd.Categorical(logits=logits)

print(f'Categorical(logits={logits.tolist()}):')
print(f'  batch_shape={cat.batch_shape}, event_shape={cat.event_shape}')
print(f'  probs = {cat.probs_parameter().round(3)}')
print(f'  log_prob(0) = {cat.log_prob(0):.4f}  '
      f'(prob={jnp.exp(cat.log_prob(0)):.4f})')
print(f'  log_prob(4) = {cat.log_prob(4):.4f}  '
      f'(prob={jnp.exp(cat.log_prob(4)):.4f})')
print()

key, subkey = jax.random.split(key)
s = cat.sample(10, seed=subkey)
print(f'  sample(10) = {s}')
print(f'  dtype      = {s.dtype}  (integer)')
print()

# Batch of Categoricals — batch_shape=(3,), each with 4 classes
batch_logits = jnp.array([[2., 1., 0., -1.],
                           [0., 0., 0., 0.],
                           [-1., 0., 1., 2.]])
cat_batch = tfd.Categorical(logits=batch_logits)
print(f'Batch of 3 Categoricals (4 classes each):')
print(f'  batch_shape={cat_batch.batch_shape}, event_shape={cat_batch.event_shape}')
labels = jnp.array([0, 1, 3])  # one label per distribution in batch
print(f'  log_prob([0,1,3]).shape = {cat_batch.log_prob(labels).shape}  # (3,)')
print(f'  log_prob values = {cat_batch.log_prob(labels).round(3)}')
print()

# One-hot distribution (Categorical variant over one-hot vectors)
# event_shape = (k,) instead of ()
oh = tfd.OneHotCategorical(logits=logits)
print(f'OneHotCategorical: event_shape={oh.event_shape}')
key, subkey = jax.random.split(key)
print(f'  sample: {oh.sample(seed=subkey)}')

## 2.5 Empirical Distribution

`tfd.Empirical` represents the discrete uniform distribution over a finite set of
provided samples:

$$
P(X = x) = \frac{1}{N} \sum_{n=1}^{N} \mathbf{1}[x = x_n]
$$

This is the **empirical measure** / empirical distribution of a dataset.

### Important semantics
- `log_prob(x)` returns $\log(k/N)$ where $k$ is the number of times $x$
  appears in the stored samples — this is only well-defined for **discrete** data
  or when $x$ is exactly one of the stored points.
- For **continuous** empirical distributions (kernel density estimation), use
  `tfd.MixtureSameFamily` with Gaussian kernels.
- `sample(n, seed)` samples **with replacement** from the stored points.
- The `event_ndims` constructor argument controls how many trailing dimensions
  of `samples` constitute the event shape.

### Use cases
- Importance-weighted distributions (with non-uniform weights: see `tfd.FiniteDiscrete`)
- Particle approximations in particle filtering
- Bootstrapping / empirical Bayes

In [ ]:
# ── 2.5  Empirical distribution ───────────────────────────────────────────────

# Scalar empirical distribution
data_1d = jnp.array([1., 2., 2., 3., 4., 4., 4.])
emp_scalar = tfd.Empirical(samples=data_1d, event_ndims=0)

print(f'Empirical (scalar, N=7):')
print(f'  batch_shape={emp_scalar.batch_shape}, event_shape={emp_scalar.event_shape}')
print(f'  mean()     = {emp_scalar.mean():.4f}  (expected {data_1d.mean():.4f})')
print(f'  variance() = {emp_scalar.variance():.4f}  (expected {data_1d.var():.4f})')
print(f'  log_prob(4.0) = {emp_scalar.log_prob(4.):.4f}  '
      f'(3/7 ≈ {3/7:.4f}, log={jnp.log(3/7):.4f})')
print(f'  log_prob(1.0) = {emp_scalar.log_prob(1.):.4f}  '
      f'(1/7 ≈ {1/7:.4f}, log={jnp.log(1/7):.4f})')
print()

key, subkey = jax.random.split(key)
boot_samples = emp_scalar.sample(10, seed=subkey)
print(f'  sample(10) [bootstrap] = {boot_samples}')
print()

# Vector empirical distribution  (event_ndims=1)
key, k1 = jax.random.split(key)
data_2d = jax.random.normal(k1, (50, 3))   # 50 points in R^3
emp_vec = tfd.Empirical(samples=data_2d, event_ndims=1)
print(f'Empirical (vector R^3, N=50):')
print(f'  batch_shape={emp_vec.batch_shape}, event_shape={emp_vec.event_shape}')
print(f'  mean() = {emp_vec.mean().round(3)}')
key, subkey = jax.random.split(key)
print(f'  sample(3).shape = {emp_vec.sample(3, seed=subkey).shape}')
print()

# For continuous KDE, use MixtureSameFamily:
bandwidth = 0.3
kde = tfd.MixtureSameFamily(
    mixture_distribution=tfd.Categorical(probs=jnp.ones(len(data_1d)) / len(data_1d)),
    components_distribution=tfd.Normal(loc=data_1d, scale=bandwidth),
)
print(f'KDE (Gaussian, bw={bandwidth}):')
print(f'  log_prob(3.0) = {kde.log_prob(3.0):.4f}  (continuous, always finite)')

## 2.6 Continuous vs. Discrete Distributions

| Property | Continuous | Discrete |
|----------|-----------|---------|
| `log_prob(x)` | Log probability *density* (can be > 0) | Log probability *mass* (≤ 0) |
| `sample` dtype | `float32` / `float64` | `int32` / `int64` |
| `cdf` / `quantile` | Well-defined | Step function |
| `entropy` | Differential entropy (can be $-\infty$) | Shannon entropy |

TFP does not enforce this at the type level — the distinction is captured by
the distribution's `is_continuous` property and documented per-class.

**Mixed distributions** (e.g., Spike-and-slab, zero-inflated) can be constructed
with `tfd.MixtureSameFamily` or `tfd.Mixture`, or by subclassing `tfd.Distribution`
and implementing `_log_prob` directly.

## 2.7 Parameter Constraints and `parameter_properties()`

Every TFP distribution class exposes a class method
`parameter_properties(dtype=tf.float32, num_classes=None)` (source:
`distribution.py`) that describes the constraints and default bijectors for
each parameter.

For example, `tfd.Normal.parameter_properties()` returns:
```python
{
  'loc':   ParameterProperties(),            # unconstrained
  'scale': ParameterProperties(
               default_constraining_bijector_fn=lambda: tfb.Softplus()
           ),                                # must be positive
}
```

The `default_constraining_bijector_fn` is the bijector used to map from
unconstrained space to the valid parameter domain.  This is used by
`tfd.JointDistributionCoroutineAutoBatched` and TFP's variational inference
utilities to automatically construct unconstrained parameterisations.

### How constraints are applied in practice

When you need to *optimise* distribution parameters (e.g., in VI), you work in
unconstrained space and pass through the constraining bijector:

```python
# Optimise a Normal's scale in log-space
log_scale_free = jnp.array(0.0)             # unconstrained
scale = tfb.Softplus()(log_scale_free)      # positive
dist = tfd.Normal(loc=0., scale=scale)
```

In [ ]:
# ── 2.7  Parameter constraints and properties ─────────────────────────────────

# Inspect parameter properties for several distributions
for cls in [tfd.Normal, tfd.Gamma, tfd.Beta, tfd.Dirichlet]:
    props = cls.parameter_properties()
    print(f'{cls.__name__}:')
    for name, prop in props.items():
        bij_name = None
        try:
            bij_name = type(prop.default_constraining_bijector_fn()).__name__
        except Exception:
            bij_name = '(identity / unconstrained)'
        print(f'  {name:20s}  constraining bijector: {bij_name}')
    print()

# ── Unconstrained optimisation of distribution parameters ─────────────────────
print('='*60)
print('Optimising a Gamma distribution in unconstrained space:')

# Gamma(concentration=alpha, rate=beta), both must be > 0.
# We optimise log(alpha) and log(beta) (softplus^{-1} is an alternative).

# True distribution
true_gamma = tfd.Gamma(concentration=3., rate=0.5)
key, subkey = jax.random.split(key)
obs = true_gamma.sample(500, seed=subkey)

# Loss: negative log-likelihood in unconstrained space
softplus = tfb.Softplus()

def neg_log_lik(log_alpha, log_beta):
    alpha = softplus.forward(log_alpha)   # ensures > 0
    beta  = softplus.forward(log_beta)
    return -tfd.Gamma(alpha, beta).log_prob(obs).mean()

# Simple gradient descent
log_a = jnp.array(0.0)
log_b = jnp.array(0.0)
lr = 0.05
grad_fn = jax.value_and_grad(neg_log_lik, argnums=(0, 1))

for step in range(300):
    loss, (ga, gb) = grad_fn(log_a, log_b)
    log_a = log_a - lr * ga
    log_b = log_b - lr * gb

print(f'  Estimated: alpha={softplus(log_a):.3f}, beta={softplus(log_b):.3f}')
print(f'  True:      alpha=3.000, beta=0.500')

## 2.8 Distribution Class Hierarchy

The inheritance structure in TFP (simplified):

```
tfd.Distribution                          (abstract base)
├── Univariate continuous
│   ├── tfd.Normal
│   ├── tfd.Gamma, Beta, Dirichlet, ...
│   └── tfd.Uniform, Exponential, ...
├── Univariate discrete
│   ├── tfd.Categorical
│   ├── tfd.Bernoulli, Binomial, Poisson, ...
│   └── tfd.Empirical, FiniteDiscrete
├── Multivariate continuous
│   ├── tfd.MultivariateNormalDiag
│   ├── tfd.MultivariateNormalTriL
│   ├── tfd.MultivariateNormalFullCovariance
│   ├── tfd.MultivariateNormalLinearOperator
│   ├── tfd.WishartTriL (distribution over PSD matrices)
│   └── tfd.VonMises, VonMisesFisher (on spheres)
├── Meta-distributions (wrap other distributions)
│   ├── tfd.Independent       (promote batch → event dims)
│   ├── tfd.TransformedDistribution   (push-forward via bijector)
│   ├── tfd.MixtureSameFamily (mixture with shared component family)
│   └── tfd.Mixture           (mixture with heterogeneous components)
└── Joint distributions (Section 4)
    ├── tfd.JointDistributionSequential
    ├── tfd.JointDistributionNamed
    ├── tfd.JointDistributionCoroutine
    └── tfd.JointDistributionNamedAutoBatched
```

The key meta-distribution for our purposes is `tfd.TransformedDistribution`,
which is the bridge between distributions and bijectors (Section 3).

---
# 3. Bijectors

## 3.1 What is a Bijector?

A **bijector** represents a smooth, invertible function $f: \mathcal{X} \to \mathcal{Y}$.
The key operations are:

| Method | Computes |
|--------|---------|
| `forward(x)` | $y = f(x)$ |
| `inverse(y)` | $x = f^{-1}(y)$ |
| `forward_log_det_jacobian(x, event_ndims)` | $\log |\det J_f(x)|$ |
| `inverse_log_det_jacobian(y, event_ndims)` | $\log |\det J_{f^{-1}}(y)|$ |

Source: `tensorflow_probability/python/bijectors/bijector.py`

The `event_ndims` argument specifies how many trailing dimensions are treated as
a single event (the Jacobian determinant is computed over those dimensions).

## 3.2 The Change-of-Variables Formula

If $X \sim p_X$ and $Y = f(X)$, the density of $Y$ is:

$$
\log p_Y(y) = \log p_X\!\left(f^{-1}(y)\right) + \log \left|\det J_{f^{-1}}(y)\right|
$$

where $J_{f^{-1}}$ is the Jacobian of $f^{-1}$.

Equivalently (using the inverse function theorem $J_{f^{-1}} = J_f^{-1}$):

$$
\log p_Y(y) = \log p_X\!\left(f^{-1}(y)\right) - \log \left|\det J_f\!\left(f^{-1}(y)\right)\right|
$$

**Sampling**: $y = f(x)$ where $x \sim p_X$ — use `forward`.
**Density evaluation**: requires `inverse` and `inverse_log_det_jacobian`.

## 3.3 The `Bijector` Base Class

Concrete bijectors implement private methods:
```python
_forward(x)
_inverse(y)
_forward_log_det_jacobian(x, event_ndims)   # OR
_inverse_log_det_jacobian(y, event_ndims)   # at least one must be implemented
```

Two important class attributes:
- `forward_min_event_ndims`: minimum `event_ndims` for which `forward` is defined
  (0 for elementwise bijectors, 1 for vector bijectors, 2 for matrix bijectors)
- `is_constant_jacobian`: `True` for affine maps — enables optimisations

## 3.4 Composing Bijectors with `Chain`

`tfb.Chain([b1, b2, ..., bN])` applies bijectors in **right-to-left** order
(like mathematical function composition):

$$
\text{Chain}([b_1, b_2, \ldots, b_N])(x) = b_1(b_2(\cdots b_N(x)\cdots))
$$

The log-Jacobian of a composition is the sum of individual log-Jacobians:

$$
\log|\det J_{\text{Chain}}(x)| = \sum_{k=1}^{N} \log|\det J_{b_k}(x_k)|
$$

where $x_k$ is the input to bijector $b_k$.

## 3.5 `TransformedDistribution`

The bridge between distributions and bijectors:

```python
tfd.TransformedDistribution(distribution=base_dist, bijector=f)
```

This represents the pushforward measure $p_Y = f_* p_X$.

In [ ]:
# ── 3.1–3.3  Bijector basics ─────────────────────────────────────────────────

print('Basic bijector operations:')
print('='*60)

# Exp bijector: f(x) = exp(x)
exp_bij = tfb.Exp()
x_val = jnp.array([-1., 0., 1., 2.])

print(f'Exp bijector:')
print(f'  forward([-1,0,1,2])              = {exp_bij.forward(x_val).round(4)}')
print(f'  inverse([0.37,1,2.72,7.39])      = {exp_bij.inverse(exp_bij.forward(x_val)).round(4)}')
print(f'  fwd_log_det_jacobian (event_ndims=0) = {exp_bij.forward_log_det_jacobian(x_val, 0).round(4)}')
print(f'  (should equal x since d/dx exp(x) = exp(x), log|exp(x)|=x)')
print()

# Affine bijector: f(x) = scale * x + shift
# In modern TFP, use Chain([Shift, Scale])
shift_bij = tfb.Shift(3.0)
scale_bij = tfb.Scale(2.0)
affine = tfb.Chain([shift_bij, scale_bij])   # f(x) = 2x + 3

print(f'Affine Chain([Shift(3), Scale(2)]): f(x) = 2x + 3')
print(f'  forward([0,1,2]) = {affine.forward(jnp.array([0.,1.,2.]))}')
print(f'  inverse([3,5,7]) = {affine.inverse(jnp.array([3.,5.,7.]))}')
# log|det J| = log(2) for scalar affine
print(f'  fwd_log_det_jacobian = {affine.forward_log_det_jacobian(jnp.array(0.), 0):.4f}')
print(f'  (should be log(2) = {jnp.log(2.):.4f})')
print()

# Softplus: f(x) = log(1 + exp(x)) -- smooth positive-valued bijector
sp = tfb.Softplus()
x_test = jnp.linspace(-3., 3., 5)
print(f'Softplus: f(x) = log(1+exp(x))')
print(f'  forward({x_test.round(1)}) = {sp.forward(x_test).round(4)}')
print(f'  inverse({sp.forward(x_test).round(4)}) = {sp.inverse(sp.forward(x_test)).round(4)}')
print()

# Sigmoid: f(x) = 1/(1+exp(-x)) -- maps R -> (0,1)
sig = tfb.Sigmoid()
print(f'Sigmoid: f(x) = 1/(1+exp(-x))')
print(f'  forward({x_test.round(1)}) = {sig.forward(x_test).round(4)}')

In [ ]:
# ── 3.4  Composing bijectors ─────────────────────────────────────────────────

print('Composing bijectors:')
print('='*60)

# Chain applies RIGHT-TO-LEFT (last element first)
# Chain([b1, b2]) means: first apply b2, then b1
# i.e., Chain([b1, b2])(x) = b1(b2(x))

b = tfb.Chain([tfb.Exp(), tfb.Shift(1.0)])
# b(x) = exp(x + 1)
x_in = jnp.array(0.)
print(f'Chain([Exp(), Shift(1.0)]) applied to x=0:')
print(f'  Expected: exp(0 + 1) = {jnp.exp(1.):.4f}')
print(f'  Got:      {b.forward(x_in):.4f}')
print()

# Verify log det Jacobian
# d/dx exp(x+1) = exp(x+1), so log|J| = x+1
print(f'  fwd_log_det_jacobian(0, 0) = {b.forward_log_det_jacobian(x_in, 0):.4f}')
print(f'  Expected: 1.0 (= x+1 evaluated at x=0)')
print()

# Matrix bijector: CholeskyOuterProduct maps lower-triangular L -> L @ L.T
# This is used internally by MultivariateNormalTriL
chol_bij = tfb.CholeskyOuterProduct()
L_mat = jnp.array([[2., 0.], [1., 3.]])
Sigma = chol_bij.forward(L_mat)
print(f'CholeskyOuterProduct: L -> L @ L.T')
print(f'  L = \n{L_mat}')
print(f'  Sigma = L @ L.T =\n{Sigma}')
print(f'  Recovered L = \n{chol_bij.inverse(Sigma).round(6)}')
print()

# SoftmaxCentered: maps R^{k-1} -> probability simplex in R^k
softmax_c = tfb.SoftmaxCentered()
x_simplex = jnp.array([0., 0.])     # in R^2, maps to R^3 simplex
print(f'SoftmaxCentered: R^2 -> probability simplex in R^3')
print(f'  forward([0.,0.]) = {softmax_c.forward(x_simplex).round(4)}')
print(f'  (sums to {softmax_c.forward(x_simplex).sum():.4f})')

In [ ]:
# ── 3.5  TransformedDistribution ─────────────────────────────────────────────

print('TransformedDistribution: pushforward distributions')
print('='*60)

# Log-Normal: if X ~ Normal(mu, sigma), then Y = exp(X) ~ LogNormal(mu, sigma)
normal_base = tfd.Normal(loc=1., scale=0.5)
log_normal  = tfd.TransformedDistribution(
    distribution=normal_base,
    bijector=tfb.Exp(),
)

print(f'LogNormal via TransformedDistribution:')
print(f'  event_shape = {log_normal.event_shape}')
y_test = jnp.array([0.5, 1., 2., 5.])
lp_custom = log_normal.log_prob(y_test)
lp_builtin = tfd.LogNormal(loc=1., scale=0.5).log_prob(y_test)
print(f'  log_prob matches LogNormal builtin: {jnp.allclose(lp_custom, lp_builtin)}')
print()

# Truncated Normal via Sigmoid transform (approximate):
# A proper Truncated Normal is available as tfd.TruncatedNormal,
# but illustrating the pattern:
normal_std = tfd.Normal(loc=0., scale=1.)
# Map to (0,1) via sigmoid
beta_approx = tfd.TransformedDistribution(
    distribution=normal_std,
    bijector=tfb.Sigmoid(),
)
print(f'Normal pushed through Sigmoid (≈Beta-like on (0,1)):')
print(f'  sample(5) = {beta_approx.sample(5, seed=key).round(4)}')
print(f'  (all values in (0,1): {bool(jnp.all(beta_approx.sample(5,seed=key) > 0))})')
print()

# Multivariate: Normal -> MultivariateNormal via scale_tril
# This is how MultivariateNormalTriL is implemented internally!
std_normal = tfd.Independent(
    tfd.Normal(loc=jnp.zeros(3), scale=1.),
    reinterpreted_batch_ndims=1,
)
mu_vec = jnp.array([1., 2., 3.])
L_mat  = jnp.array([[2., 0., 0.],
                     [1., 3., 0.],
                     [0., 0.5, 1.]])
affine_bij = tfb.Shift(mu_vec)(tfb.ScaleMatvecTriL(L_mat))
mvn_transformed = tfd.TransformedDistribution(
    distribution=std_normal,
    bijector=affine_bij,
)
mvn_direct = tfd.MultivariateNormalTriL(loc=mu_vec, scale_tril=L_mat)
x_test3 = jnp.zeros(3)
print(f'MVN via TransformedDistribution vs direct:')
print(f'  log_prob matches: {jnp.allclose(mvn_transformed.log_prob(x_test3),
                                           mvn_direct.log_prob(x_test3), atol=1e-5)}')

## 3.6 Important Bijectors Reference

| Bijector | Forward map $f(x)$ | Notes |
|----------|--------------------|-------|
| `Exp` | $e^x$ | Positivity constraint |
| `Log` | $\log x$ | Inverse of Exp |
| `Softplus` | $\log(1+e^x)$ | Smooth positive constraint |
| `Sigmoid` | $\sigma(x) = (1+e^{-x})^{-1}$ | Maps to $(0,1)$ |
| `Logit` | $\log x/(1-x)$ | Inverse of Sigmoid |
| `Shift(b)` | $x + b$ | Translation |
| `Scale(a)` | $ax$ | Scaling |
| `ScaleMatvecTriL(L)` | $Lx$ | Lower-triangular matrix multiply |
| `ScaleMatvecDiag(d)` | $\text{diag}(d) x$ | Diagonal scaling |
| `CholeskyOuterProduct` | $L \mapsto LL^\top$ | PSD constraint |
| `CholeskyToInvCholesky` | $L \mapsto L^{-\top}$ | Precision parameterisation |
| `SoftmaxCentered` | $\mathbb{R}^{k-1} \to \Delta^k$ | Probability simplex |
| `Permute(perm)` | reorder elements | Permutation |
| `Reshape(shape_in, shape_out)` | reshape | Shape change |
| `Chain([b1,...,bN])` | $b_1 \circ \cdots \circ b_N$ | Composition |
| `Invert(b)` | $b^{-1}$ | Swap forward/inverse |
| `RealNVP` | coupling layer | Normalizing flow block |
| `MaskedAutoregressiveFlow` | autoregressive | Normalizing flow block |

### Normalizing Flows (brief preview)

Normalizing flows use a sequence of bijectors to construct flexible distributions
from a simple base (e.g., standard Gaussian):

```python
flow = tfd.TransformedDistribution(
    distribution=tfd.Normal(0., 1.),
    bijector=tfb.Chain([
        tfb.MaskedAutoregressiveFlow(shift_and_log_scale_fn=made_network),
        tfb.Permute([1, 0]),
        tfb.MaskedAutoregressiveFlow(shift_and_log_scale_fn=made_network),
    ]),
)
```
Each bijector must be differentiable and invertible; the log-likelihood is
tractable via the chain rule for Jacobians.

---
# 4. Joint Distributions

## 4.1 Design Philosophy

Joint distributions in TFP represent a *directed graphical model* — a collection
of random variables with a defined dependency structure.  The key design goals:

1. **Declarative model specification** — describe the generative process; TFP infers
   the dependency graph automatically.
2. **Named components** — variables can be accessed by name, not just position.
3. **Composability** — joint distributions are themselves `Distribution` objects
   and can be nested.
4. **Conditioning** — any subset of variables can be "pinned" to observed values.

## 4.2 The Four Flavours

| Class | Specification style | Named? | Auto-batch? |
|-------|--------------------:|--------|-------------|
| `JointDistributionSequential` (JDS) | Python list | No (positional) | No |
| `JointDistributionNamed` (JDN) | Python dict | Yes | No |
| `JointDistributionCoroutine` (JDC) | Generator function | Yes (yield name) | No |
| `JointDistributionNamedAutoBatched` (JDNAB) | dict or coroutine | Yes | **Yes** |

**Recommendation**: Use `JointDistributionNamed` or `JointDistributionCoroutine`
for most work.  Use `JointDistributionNamedAutoBatched` when you want automatic
broadcasting of batch dimensions (removes many shape headaches).

## 4.3 Internal Storage and Dependency Resolution

In all flavours, component distributions can depend on previously sampled values.
Dependencies are expressed as **callables**:

```python
# The second component is a function of the first's sample
model = tfd.JointDistributionNamed({
    'z': tfd.Normal(0., 1.),
    'x': lambda z: tfd.Normal(z, 0.5),   # x depends on z
})
```

TFP resolves the dependency graph by:
1. Inspecting each callable's argument names using Python's `inspect` module.
2. Matching argument names against previously defined variable names.
3. Calling each callable with the relevant already-sampled values.

**Canonical order**:
- `JDS`: list insertion order
- `JDN`: dict insertion order (Python 3.7+ guarantees this)
- `JDC`: `yield` statement order in the generator

## 4.4 Accessing Marginals

TFP joint distributions do **not** compute analytical marginals.  To obtain
samples from a marginal, simply sample the joint and extract the component:

```python
samples = jd.sample(1000, seed=key)
z_marginal_samples = samples['z']   # JDN
z_marginal_samples = samples[0]     # JDS
```

For conditional distributions (given observations), use `experimental_pin` (§4.6).

In [ ]:
# ── 4.2  JointDistributionSequential ─────────────────────────────────────────

print('JointDistributionSequential:')
print('='*60)

# A simple Bayesian hierarchical model:
#   mu  ~ Normal(0, 1)         (prior on mean)
#   tau ~ HalfNormal(1)        (prior on precision)
#   x   ~ Normal(mu, 1/tau)    (likelihood)
jds = tfd.JointDistributionSequential([
    tfd.Normal(loc=0., scale=1.),               # mu
    tfd.HalfNormal(scale=1.),                   # tau  (must be positive)
    lambda tau, mu: tfd.Normal(loc=mu,          # x | mu, tau
                               scale=1./tau),
])
# Note: in JDS, the lambda arguments are matched BACKWARDS from the list.
# The last-defined element is the first argument.

print(f'  Structure: mu, tau, x')
key, subkey = jax.random.split(key)
sample = jds.sample(seed=subkey)
print(f'  sample() = {[float(s) for s in sample]}  [mu, tau, x]')
mu_s, tau_s, x_s = sample
print(f'  log_prob(sample) = {jds.log_prob(sample):.4f}')
print()

# batch sampling
key, subkey = jax.random.split(key)
batch = jds.sample(5, seed=subkey)
print(f'  sample(5) shapes: {[s.shape for s in batch]}')

In [ ]:
# ── 4.3  JointDistributionNamed ───────────────────────────────────────────────

print('JointDistributionNamed:')
print('='*60)

# Same model, but with named components and clearer argument matching.
# In JDN, lambda argument names must match previously-defined keys.
jdn = tfd.JointDistributionNamed({
    'mu':  tfd.Normal(loc=0., scale=1.),
    'tau': tfd.HalfNormal(scale=1.),
    'x':   lambda mu, tau: tfd.Normal(loc=mu, scale=1./tau),
})

key, subkey = jax.random.split(key)
samples = jdn.sample(seed=subkey)
print(f'  Single sample (named): {samples}')
print(f'  Keys: {list(samples.keys())}')
print()

key, subkey = jax.random.split(key)
batch_samples = jdn.sample(1000, seed=subkey)
print(f'  Batch sample shapes: mu={batch_samples["mu"].shape}, '
      f'tau={batch_samples["tau"].shape}, x={batch_samples["x"].shape}')
print()

# log_prob accepts a dict
lp = jdn.log_prob({'mu': 0., 'tau': 1., 'x': 0.5})
print(f'  log_prob(mu=0, tau=1, x=0.5) = {lp:.4f}')
# Decompose into components
lp_mu  = tfd.Normal(0., 1.).log_prob(0.)
lp_tau = tfd.HalfNormal(1.).log_prob(1.)
lp_x   = tfd.Normal(0., 1./1.).log_prob(0.5)
print(f'  Decomposed: {lp_mu:.4f} + {lp_tau:.4f} + {lp_x:.4f} = {lp_mu+lp_tau+lp_x:.4f}')

In [ ]:
# ── 4.4  JointDistributionCoroutine ──────────────────────────────────────────

print('JointDistributionCoroutine:')
print('='*60)

# The coroutine style is most flexible — supports complex control flow.
# Root() wraps root nodes (nodes with no parents).
Root = tfd.JointDistributionCoroutine.Root

@tfd.JointDistributionCoroutine
def jdc():
    mu  = yield Root(tfd.Normal(loc=0., scale=1.),       name='mu')
    tau = yield Root(tfd.HalfNormal(scale=1.),           name='tau')
    x   = yield tfd.Normal(loc=mu, scale=1./tau,         name='x')

key, subkey = jax.random.split(key)
s = jdc.sample(seed=subkey)
print(f'  Sample type: {type(s).__name__}')
print(f'  Sample fields: {s._fields}')         # namedtuple
print(f'  mu={float(s.mu):.3f}, tau={float(s.tau):.3f}, x={float(s.x):.3f}')
print()

# A more complex model: hierarchical with plates
print('More complex model: Bayesian linear regression')

n_obs = 10
x_obs = jnp.linspace(-2., 2., n_obs)

@tfd.JointDistributionCoroutine
def blr_model():
    slope     = yield Root(tfd.Normal(0., 1.),       name='slope')
    intercept = yield Root(tfd.Normal(0., 2.),       name='intercept')
    sigma     = yield Root(tfd.HalfNormal(0.5),      name='sigma')
    # Plate: n_obs observations
    y = yield tfd.Independent(
        tfd.Normal(loc=slope * x_obs + intercept, scale=sigma),
        reinterpreted_batch_ndims=1,
        name='y',
    )

key, subkey = jax.random.split(key)
s_blr = blr_model.sample(seed=subkey)
print(f'  slope={float(s_blr.slope):.3f}, '
      f'intercept={float(s_blr.intercept):.3f}, '
      f'sigma={float(s_blr.sigma):.3f}')
print(f'  y shape: {s_blr.y.shape}')
print(f'  log_prob = {blr_model.log_prob(s_blr):.4f}')

In [ ]:
# ── 4.5  JointDistributionNamedAutoBatched ───────────────────────────────────

print('JointDistributionNamedAutoBatched (JDNAB):')
print('='*60)
# JDNAB automatically handles batch dimensions.
# You write the model as if for a single observation; JDNAB vmaps internally.

n_obs = 20
x_obs = jnp.linspace(-2., 2., n_obs)
key, k1, k2 = jax.random.split(key, 3)
y_obs = 2. * x_obs + 1. + 0.2 * jax.random.normal(k1, (n_obs,))

jdnab = tfd.JointDistributionNamedAutoBatched({
    'slope':     tfd.Normal(0., 1.),
    'intercept': tfd.Normal(0., 2.),
    'y':         lambda slope, intercept: tfd.Normal(
                     slope * x_obs + intercept, 0.2
                 ),
})

# Sample a single draw
s = jdnab.sample(seed=k2)
print(f'  Sample: slope={float(s["slope"]):.3f}, intercept={float(s["intercept"]):.3f}')
print(f'  y shape: {s["y"].shape}')

# log_prob of observations (pin y=y_obs, then log_prob over slope and intercept)
pinned = jdnab.experimental_pin(y=y_obs)
print(f'\nPinned model (y fixed to observations):')
print(f'  log_prob(slope=2, intercept=1) = '
      f'{pinned.log_prob(slope=jnp.array(2.), intercept=jnp.array(1.)):.4f}')
print(f'  log_prob(slope=0, intercept=0) = '
      f'{pinned.log_prob(slope=jnp.array(0.), intercept=jnp.array(0.)):.4f}')

In [ ]:
# ── 4.6  Pinning and conditioning ─────────────────────────────────────────────

print('Pinning / conditioning a joint distribution:')
print('='*60)

# JDN model
jdn_model = tfd.JointDistributionNamed({
    'z': tfd.Normal(loc=0., scale=1.),
    'x': lambda z: tfd.Normal(loc=z, scale=0.5),
})

# Pin x to observed value x_obs = 1.5
x_observed = 1.5
jdn_pinned = jdn_model.experimental_pin(x=x_observed)

print(f'Original model:')
print(f'  Variables: z, x')
print(f'  log_prob(z=0, x=1.5) = {jdn_model.log_prob({"z": 0., "x": x_observed}):.4f}')
print()

print(f'Pinned model (x = {x_observed}):')
print(f'  Remaining variables: z only')
# The pinned model's log_prob takes only the free variables
lp_pinned = jdn_pinned.log_prob({'z': jnp.array(0.)})
print(f'  log_prob(z=0) = {lp_pinned:.4f}')
# This equals log p(z=0) + log p(x=1.5 | z=0)
manual = (tfd.Normal(0.,1.).log_prob(0.) +
          tfd.Normal(0., 0.5).log_prob(x_observed))
print(f'  Manual check: {manual:.4f}')
print()

# The pinned model can be used directly with MCMC
@jax.jit
def posterior_sample(seed):
    kernel = tfm.HamiltonianMonteCarlo(
        target_log_prob_fn=lambda z: jdn_pinned.log_prob({'z': z}),
        step_size=0.3,
        num_leapfrog_steps=5,
    )
    z_samples, _ = tfm.sample_chain(
        num_results=1000,
        current_state=jnp.array(0.),
        kernel=kernel,
        num_burnin_steps=200,
        seed=seed,
    )
    return z_samples

key, subkey = jax.random.split(key)
z_posterior = posterior_sample(subkey)
# Analytical posterior: z|x ~ N( x/(1+sigma_z^2/sigma_x^2), ... )
# With sigma_z=1, sigma_x=0.5: posterior mean = x/(1+0.25) = 1.5/1.25 = 1.2
print(f'Posterior z | x=1.5 (analytical mean = 1.2):')
print(f'  MCMC estimate: {z_posterior.mean():.3f} ± {z_posterior.std():.3f}')

---
# 5. Conditional Gaussian Example

## 5.1 Problem Setup

Let $(X, Y)$ be a jointly Gaussian random vector:

$$
\begin{pmatrix} X \\ Y \end{pmatrix} \sim \mathcal{N}\!\left(
\begin{pmatrix} \mu_X \\ \mu_Y \end{pmatrix},\;
\begin{pmatrix} \Sigma_{XX} & \Sigma_{XY} \\ \Sigma_{YX} & \Sigma_{YY} \end{pmatrix}
\right)
$$

where $X \in \mathbb{R}^{d_X}$ and $Y \in \mathbb{R}^{d_Y}$.

## 5.2 Analytical Conditional Distribution

By the Schur complement formula:

$$
X \mid Y = y \;\sim\; \mathcal{N}\!\left(\,\mu_{X|Y},\; \Sigma_{X|Y}\right)
$$

where:

$$
\mu_{X|Y} = \mu_X + \Sigma_{XY}\,\Sigma_{YY}^{-1}(y - \mu_Y)
$$

$$
\Sigma_{X|Y} = \Sigma_{XX} - \Sigma_{XY}\,\Sigma_{YY}^{-1}\,\Sigma_{YX}
$$

This is the **Schur complement** of $\Sigma_{YY}$ in $\Sigma$.

## 5.3 Matheron's Rule (Pathwise Conditioning)

Matheron's rule provides an alternative way to *sample* from $X|Y=y$ without
explicitly forming the conditional distribution.

**Theorem** (Matheron's rule): Draw a joint sample $(X^*, Y^*)$ from the
unconditional distribution.  Then:

$$
\tilde{X} \;=\; X^* + \Sigma_{XY}\,\Sigma_{YY}^{-1}(y - Y^*)
\;\sim\; X \mid Y = y
$$

**Proof**:
- $\tilde{X}$ is Gaussian (linear function of Gaussians).
- $\mathbb{E}[\tilde{X}] = \mu_X + \Sigma_{XY}\Sigma_{YY}^{-1}(y - \mu_Y) = \mu_{X|Y}$ ✓
- $\text{Cov}(\tilde{X}) = \text{Cov}(X^* - \Sigma_{XY}\Sigma_{YY}^{-1}Y^*)$
  $= \Sigma_{XX} - \Sigma_{XY}\Sigma_{YY}^{-1}\Sigma_{YX} = \Sigma_{X|Y}$ ✓

**Why use Matheron's rule?**
1. **Avoids Cholesky of conditional covariance** — only requires solving a system
   with $\Sigma_{YY}$ and sampling from the joint.
2. **Gradient-friendly** — the reparameterisation trick flows through the linear
   correction term naturally.
3. **Extensible** — generalises to non-Gaussian conditionals when combined with
   approximate methods (e.g., in deep GPs).

## 5.4 Low-Rank Structure

When $\Sigma_{YY} = H\Sigma_{XX}H^\top + \sigma^2 I$ (common in GP regression
with a linear observation model), the Woodbury identity gives:

$$
\Sigma_{YY}^{-1} = \frac{1}{\sigma^2}\left[I - H\left(\Sigma_{XX}^{-1} + \frac{1}{\sigma^2}H^\top H\right)^{-1}\frac{H^\top}{\sigma^2}\right]
$$

TFP supports this via `tf.linalg.LinearOperatorLowRankUpdate`, which exploits
the structure for $O(d_X^2 d_Y + d_Y^3)$ instead of $O(d_Y^3)$ solves when
$d_X \ll d_Y$.

In [ ]:
# ── 5.2  Analytical Gaussian conditional ──────────────────────────────────────

def gaussian_conditional(mu_X, mu_Y, Sigma_XX, Sigma_XY, Sigma_YY, y_obs):
    # Compute the parameters of X | Y=y_obs analytically.
    # Returns mu_cond, Sigma_cond
    # Solve Sigma_YY @ K = Sigma_YX  =>  K = Sigma_YY^{-1} Sigma_YX
    # Then Sigma_XY @ K = Sigma_XY Sigma_YY^{-1} Sigma_YX
    Sigma_YX = Sigma_XY.T
    K = jnp.linalg.solve(Sigma_YY, Sigma_YX)     # (d_Y, d_X)
    mu_cond    = mu_X + Sigma_XY @ K.T @ (y_obs - mu_Y)   # wrong shape, fix:
    # Sigma_XY @ K is (d_X, d_Y) @ (d_Y, d_X) = (d_X, d_X)
    mu_cond    = mu_X + Sigma_XY @ jnp.linalg.solve(Sigma_YY, y_obs - mu_Y)
    Sigma_cond = Sigma_XX - Sigma_XY @ jnp.linalg.solve(Sigma_YY, Sigma_YX)
    return mu_cond, Sigma_cond

# ── Construct a joint Gaussian ────────────────────────────────────────────────

d_X, d_Y = 3, 2

key, k1 = jax.random.split(key)
# Build a random PSD joint covariance
d_total = d_X + d_Y
A = jax.random.normal(k1, (d_total, d_total)) / jnp.sqrt(d_total)
Sigma_joint = A @ A.T + 2.0 * jnp.eye(d_total)   # PSD

mu_joint = jnp.zeros(d_total)

# Partition: X is first d_X, Y is last d_Y
Sigma_XX = Sigma_joint[:d_X, :d_X]
Sigma_XY = Sigma_joint[:d_X, d_X:]
Sigma_YX = Sigma_joint[d_X:, :d_X]
Sigma_YY = Sigma_joint[d_X:, d_X:]
mu_X     = mu_joint[:d_X]
mu_Y     = mu_joint[d_X:]

# Joint distribution (using full covariance)
joint_dist = tfd.MultivariateNormalFullCovariance(
    loc=mu_joint,
    covariance_matrix=Sigma_joint,
)

# Analytical conditional
y_obs = jnp.array([1., -0.5])
mu_cond, Sigma_cond = gaussian_conditional(mu_X, mu_Y, Sigma_XX, Sigma_XY, Sigma_YY, y_obs)

cond_dist = tfd.MultivariateNormalFullCovariance(
    loc=mu_cond,
    covariance_matrix=Sigma_cond,
)
print(f'Conditional X | Y = {y_obs}:')
print(f'  mu_cond    = {mu_cond.round(4)}')
print(f'  Sigma_cond =\n{Sigma_cond.round(4)}')
print(f'  (Sigma_cond should be PSD; min eigenvalue = {jnp.linalg.eigvalsh(Sigma_cond).min():.4f})')

In [ ]:
# ── 5.3  Matheron's rule for sampling ─────────────────────────────────────────

def matheron_sample(joint_dist, d_X, Sigma_XY, Sigma_YY, y_obs, n_samples, seed):
    # Sample from X | Y=y_obs using Matheron's rule.
    # joint_dist : tfd.MultivariateNormal over (X, Y)
    # d_X        : dimension of X
    # 1. Draw joint samples
    joint_samples = joint_dist.sample(n_samples, seed=seed)  # (n, d_X + d_Y)
    X_star = joint_samples[:, :d_X]    # (n, d_X)
    Y_star = joint_samples[:, d_X:]    # (n, d_Y)

    # 2. Linear correction: X_star + Sigma_XY Sigma_YY^{-1} (y_obs - Y_star)
    # Solve Sigma_YY v = (y_obs - Y_star).T  -- shape (d_Y, n)
    residual = (y_obs[None, :] - Y_star).T    # (d_Y, n)
    v = jnp.linalg.solve(Sigma_YY, residual)  # (d_Y, n)
    correction = (Sigma_XY @ v).T              # (n, d_X)

    return X_star + correction

key, subkey = jax.random.split(key)
n_samples = 50_000
matheron_samples = matheron_sample(
    joint_dist, d_X, Sigma_XY, Sigma_YY, y_obs, n_samples, subkey
)

# Compare with analytical conditional distribution samples
key, subkey = jax.random.split(key)
analytical_samples = cond_dist.sample(n_samples, seed=subkey)

print('Matheron sampling vs. analytical conditional:')
print(f'  n_samples = {n_samples}')
print()
print(f'  Empirical mean (Matheron)    = {matheron_samples.mean(0).round(4)}')
print(f'  Empirical mean (analytical)  = {analytical_samples.mean(0).round(4)}')
print(f'  True conditional mean        = {mu_cond.round(4)}')
print()

matheron_cov = jnp.cov(matheron_samples.T)
analytic_emp_cov = jnp.cov(analytical_samples.T)
print(f'  Empirical cov (Matheron):\n{matheron_cov.round(4)}')
print(f'  True conditional cov:\n{Sigma_cond.round(4)}')
print()

max_err = jnp.abs(matheron_cov - Sigma_cond).max()
print(f'  Max covariance error: {max_err:.4f}  (should be small)')

In [ ]:
# ── 5.4  Low-rank structure: GP-style observation model ───────────────────────

print('Low-rank conditioning: Y = H @ X + noise')
print('='*60)
print()

# Model: X ~ N(0, Sigma_prior),  Y = H @ X + eps,  eps ~ N(0, sigma^2 I)
# Then:
#   Sigma_YY = H Sigma H^T + sigma^2 I   <- low-rank + diagonal
#   Sigma_XY = Sigma H^T

d_X_lr, d_Y_lr = 5, 20    # X is low-dim, Y is high-dim
sigma_noise = 0.5

key, k1, k2 = jax.random.split(key, 3)
H = jax.random.normal(k1, (d_Y_lr, d_X_lr))  # observation matrix

# Prior covariance for X
A_prior = jax.random.normal(k2, (d_X_lr, d_X_lr)) / jnp.sqrt(d_X_lr)
Sigma_prior = A_prior @ A_prior.T + jnp.eye(d_X_lr)

Sigma_XY_lr = Sigma_prior @ H.T                         # (d_X, d_Y)
# Sigma_YY = H Sigma H^T + sigma^2 I  -- exploitable via Woodbury
Sigma_YY_lr = H @ Sigma_prior @ H.T + sigma_noise**2 * jnp.eye(d_Y_lr)

# Woodbury identity for (H Sigma H^T + sigma^2 I)^{-1}:
# = (1/sigma^2)[I - H (Sigma^{-1} + H^T H / sigma^2)^{-1} H^T / sigma^2]
# This requires inverting a d_X x d_X matrix instead of d_Y x d_Y!
Sigma_prior_inv = jnp.linalg.inv(Sigma_prior)
M = Sigma_prior_inv + H.T @ H / sigma_noise**2          # (d_X, d_X) -- small!
M_inv = jnp.linalg.inv(M)

def woodbury_solve(b):
    # Solve (H Sigma H^T + sigma^2 I) x = b using Woodbury.
    # (A + UCV)^{-1} = A^{-1} - A^{-1} U (C^{-1} + V A^{-1} U)^{-1} V A^{-1}
    b_scaled = b / sigma_noise**2                        # (d_Y,)
    HtB = H.T @ b_scaled                                 # (d_X,)
    correction = H @ (M_inv @ HtB) / sigma_noise**2     # (d_Y,)
    return b_scaled - correction

# Verify Woodbury vs direct solve
y_test_lr = jax.random.normal(key, (d_Y_lr,))
direct_solve = jnp.linalg.solve(Sigma_YY_lr, y_test_lr)
woodbury_result = woodbury_solve(y_test_lr)
err = jnp.abs(direct_solve - woodbury_result).max()
print(f'  Woodbury vs direct solve max error: {err:.2e}  (d_X={d_X_lr}, d_Y={d_Y_lr})')
print()

# Conditional mean using Woodbury (no d_Y^3 Cholesky needed!)
mu_cond_woodbury = Sigma_XY_lr @ woodbury_solve(y_test_lr)
mu_cond_direct, Sigma_cond_direct = gaussian_conditional(
    jnp.zeros(d_X_lr), jnp.zeros(d_Y_lr),
    Sigma_prior, Sigma_XY_lr, Sigma_YY_lr, y_test_lr
)
print(f'  Conditional mean (Woodbury): {mu_cond_woodbury.round(4)}')
print(f'  Conditional mean (direct):   {mu_cond_direct.round(4)}')
print(f'  Match: {jnp.allclose(mu_cond_woodbury, mu_cond_direct, atol=1e-5)}')
print()

print('  Complexity comparison:')
print(f'    Direct:   O(d_Y^3) = O({d_Y_lr}^3) = O({d_Y_lr**3})')
print(f'    Woodbury: O(d_X^3 + d_X^2 d_Y) = O({d_X_lr**3 + d_X_lr**2*d_Y_lr})')
print()
print('  Note: tfp.math.psd_kernels and LinearOperatorLowRankUpdate can')
print('  automate this structure exploitation in GP posterior computations.')

---
# 6. MCMC in TFP

## 6.1 Design Philosophy

TFP's MCMC module (`tfp.mcmc`) is built around the **`TransitionKernel`** abstraction.
Source: `tensorflow_probability/python/mcmc/kernel.py`

### Core abstraction: `TransitionKernel`

```python
class TransitionKernel(abc.ABC):
    @abc.abstractmethod
    def one_step(self, current_state, previous_kernel_results, seed=None):
        # Returns: (next_state, kernel_results)
        ...

    @abc.abstractmethod
    def bootstrap_results(self, init_state):
        # Returns: initial kernel_results (e.g., step size, acceptance info)
        ...

    @property
    def is_calibrated(self) -> bool:
        # True if the kernel leaves the target invariant (e.g., MH-corrected)
        ...
```

### Running chains: `sample_chain`

```python
tfm.sample_chain(
    num_results,          # number of post-burnin samples to return
    current_state,        # initial state (tensor or list of tensors)
    kernel,               # a TransitionKernel
    num_burnin_steps=0,   # discarded samples at start
    num_steps_between_results=0,  # thinning
    trace_fn=...,         # callable(state, results) -> quantities to trace
    seed=None,
)
```

### Key design decisions vs. BlackJAX

| | TFP MCMC | BlackJAX |
|--|----------|---------|
| **Style** | Object-oriented kernels | Pure functions |
| **State** | Managed by `sample_chain` | Explicit; user-managed |
| **PRNG** | Passed to `sample_chain`; split internally | Explicit key per step |
| **JIT** | `sample_chain` JIT-able as a whole | Each `kernel.step` JIT-able |
| **Adaptation** | Composable wrapper kernels | Separate adaptation schedules |
| **Algorithms** | HMC, NUTS, RWM, Slice, REMC, IHMC | NUTS, HMC, MCLMC, MALA, Pathfinder, SMC, ... |
| **Integration** | Deep TFP ecosystem integration | Minimal dependencies |
| **Customisation** | Subclass `TransitionKernel` | Implement a 2-function protocol |

**TFP is better when**: you want seamless integration with TFP distributions and
joint distribution models, or you want adaptation/composition via wrapper kernels.

**BlackJAX is better when**: you want cutting-edge algorithms (MCLMC, adaptive
NUTS variants), more control over the step loop, or a lighter dependency.

## 6.2 Available Kernels

| Kernel | Description |
|--------|-------------|
| `HamiltonianMonteCarlo` | Leapfrog HMC with MH correction |
| `NoUTurnSampler` | NUTS — adaptive trajectory length |
| `RandomWalkMetropolis` | Gaussian RWM proposal |
| `MetropolisHastings` | General MH with custom proposal |
| `SliceSampler` | Slice sampling |
| `ReplicaExchangeMC` | Parallel tempering |
| `IteratedHamiltonianMonteCarlo` | IHMC with refreshed momentum |
| `UncalibratedHamiltonianMonteCarlo` | HMC without MH correction (biased) |

## 6.3 Adaptation Kernels (wrappers)

| Wrapper | Description |
|---------|-------------|
| `DualAveragingStepSizeAdaptation` | Nesterov dual averaging for step size |
| `SimpleStepSizeAdaptation` | Multiplicative step size tuning |

In [ ]:
# ── 6.2  Target distribution: a banana-shaped posterior ───────────────────────
# This is a standard MCMC test case.

def banana_log_prob(x, b=0.1):
    # Log prob of the banana distribution:
    # x[0] ~ N(0,10),  x[1]|x[0] ~ N(b*x[0]^2 - 100*b, 1)
    lp0 = tfd.Normal(0., jnp.sqrt(10.)).log_prob(x[0])
    lp1 = tfd.Normal(b * x[0]**2 - 100.*b, 1.).log_prob(x[1])
    return lp0 + lp1

# Verify it's defined
test_x = jnp.array([0., 0.])
print(f'banana_log_prob([0,0]) = {banana_log_prob(test_x):.4f}')
print(f'banana_log_prob([1,0]) = {banana_log_prob(jnp.array([1.,0.])):.4f}')

# Visualise the target
xs = jnp.linspace(-20., 20., 200)
ys = jnp.linspace(-15., 5., 200)
XX, YY = jnp.meshgrid(xs, ys)
XY = jnp.stack([XX.ravel(), YY.ravel()], axis=-1)
log_p = jax.vmap(banana_log_prob)(XY).reshape(200, 200)

plt.figure(figsize=(8, 4))
plt.contourf(XX, YY, jnp.exp(log_p), levels=30, cmap='Blues')
plt.colorbar(label='p(x)')
plt.title('Banana-shaped target distribution')
plt.xlabel('x[0]')
plt.ylabel('x[1]')
plt.tight_layout()
plt.savefig('banana_target.png', dpi=100)
plt.show()
print('Target distribution visualised.')

In [ ]:
# ── 6.3  HMC with dual-averaging step-size adaptation ─────────────────────────

print('Hamiltonian Monte Carlo with adaptation:')
print('='*60)

num_burnin  = 1000
num_samples = 3000
dim         = 2

@jax.jit
def run_hmc_chain(seed):
    # Inner kernel: leapfrog HMC
    hmc_kernel = tfm.HamiltonianMonteCarlo(
        target_log_prob_fn=banana_log_prob,
        step_size=0.3,
        num_leapfrog_steps=5,
    )
    # Outer kernel: automatically tune step size during burn-in
    adaptive_kernel = tfm.DualAveragingStepSizeAdaptation(
        inner_kernel=hmc_kernel,
        num_adaptation_steps=int(0.8 * num_burnin),
        target_accept_prob=0.75,      # target Metropolis acceptance rate
    )
    # Trace function: what to record at each step
    def trace_fn(_, pkr):
        inner = pkr.inner_results
        return {
            'is_accepted': inner.is_accepted,
            'log_accept_ratio': inner.log_accept_ratio,
            'step_size': pkr.new_step_size,
        }

    samples, trace = tfm.sample_chain(
        num_results=num_samples,
        current_state=jnp.zeros(dim),
        kernel=adaptive_kernel,
        num_burnin_steps=num_burnin,
        trace_fn=trace_fn,
        seed=seed,
    )
    return samples, trace

key, subkey = jax.random.split(key)
hmc_samples, hmc_trace = run_hmc_chain(subkey)

print(f'  Samples shape: {hmc_samples.shape}')
print(f'  Acceptance rate: {hmc_trace["is_accepted"].mean():.3f}  (target: 0.75)')
print(f'  Final step size: {hmc_trace["step_size"][-1]:.4f}')
print(f'  Sample mean: {hmc_samples.mean(0).round(4)}  (true mean ≈ [0, 0])')
print(f'  Sample std:  {hmc_samples.std(0).round(4)}')

In [ ]:
# ── 6.4  NUTS (No-U-Turn Sampler) ────────────────────────────────────────────

print('No-U-Turn Sampler (NUTS):')
print('='*60)

@jax.jit
def run_nuts_chain(seed):
    nuts_kernel = tfm.NoUTurnSampler(
        target_log_prob_fn=banana_log_prob,
        step_size=0.3,
    )
    adaptive_nuts = tfm.DualAveragingStepSizeAdaptation(
        inner_kernel=nuts_kernel,
        num_adaptation_steps=int(0.8 * num_burnin),
        target_accept_prob=0.8,   # NUTS typically targets 0.8
    )
    def trace_fn(_, pkr):
        return {
            'is_accepted':    pkr.inner_results.is_accepted,
            'step_size':      pkr.new_step_size,
            'leapfrog_count': pkr.inner_results.leapfrog_count,
        }
    samples, trace = tfm.sample_chain(
        num_results=num_samples,
        current_state=jnp.zeros(dim),
        kernel=adaptive_nuts,
        num_burnin_steps=num_burnin,
        trace_fn=trace_fn,
        seed=seed,
    )
    return samples, trace

key, subkey = jax.random.split(key)
nuts_samples, nuts_trace = run_nuts_chain(subkey)

print(f'  Acceptance rate: {nuts_trace["is_accepted"].mean():.3f}  (target: 0.8)')
print(f'  Final step size: {nuts_trace["step_size"][-1]:.4f}')
print(f'  Mean leapfrog steps/sample: {nuts_trace["leapfrog_count"].mean():.1f}')
print(f'  Sample mean: {nuts_samples.mean(0).round(4)}')
print()

# Compare ESS (effective sample size) as a rough quality metric
def ess_1d(x):
    # Rough ESS via autocorrelation for a 1-D chain.
    n = len(x)
    x_c = x - x.mean()
    acf = jnp.correlate(x_c, x_c, mode='full')[n-1:]
    acf = acf / acf[0]
    # Sum positive autocorrelations (Geyer's initial monotone sequence)
    rho_sum = 1.0
    for k in range(1, min(n//2, 500)):
        pair = acf[2*k-1] + acf[2*k]
        if pair < 0:
            break
        rho_sum += pair
    return n / (2 * rho_sum - 1)

for name, samp in [('HMC', hmc_samples), ('NUTS', nuts_samples)]:
    ess0 = ess_1d(np.array(samp[:, 0]))
    ess1 = ess_1d(np.array(samp[:, 1]))
    print(f'  {name}: ESS x[0]≈{ess0:.0f}, ESS x[1]≈{ess1:.0f}  '
          f'(out of {num_samples} samples)')

In [ ]:
# ── 6.5  Diagnostics and visualisation ───────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for row, (name, samp, trace) in enumerate([
    ('HMC',  hmc_samples,  hmc_trace),
    ('NUTS', nuts_samples, nuts_trace),
]):
    # Trace plot
    ax = axes[row, 0]
    ax.plot(samp[:300, 0], lw=0.5, alpha=0.8, label='x[0]')
    ax.plot(samp[:300, 1], lw=0.5, alpha=0.8, label='x[1]')
    ax.set_title(f'{name}: trace (first 300)')
    ax.set_xlabel('iteration')
    ax.legend(fontsize=8)

    # Sample scatter
    ax = axes[row, 1]
    ax.scatter(samp[::5, 0], samp[::5, 1], s=1, alpha=0.3)
    ax.set_title(f'{name}: samples')
    ax.set_xlabel('x[0]')
    ax.set_ylabel('x[1]')

    # Step size adaptation
    ax = axes[row, 2]
    ax.plot(trace['step_size'])
    ax.set_title(f'{name}: step size adaptation (burn-in)')
    ax.set_xlabel('burn-in iteration')
    ax.set_ylabel('step size')

plt.tight_layout()
plt.savefig('mcmc_comparison.png', dpi=100)
plt.show()
print('MCMC diagnostics plotted.')

## 6.6 Comparison with BlackJAX

[BlackJAX](https://github.com/blackjax-devs/blackjax) is a modern,
purely functional MCMC library built natively on JAX.

### Side-by-side: NUTS sampling

**TFP:**
```python
from tensorflow_probability.substrates import jax as tfp
tfm = tfp.mcmc

kernel = tfm.NoUTurnSampler(target_log_prob_fn=log_prob, step_size=0.3)
adaptive = tfm.DualAveragingStepSizeAdaptation(kernel, num_adaptation_steps=800)
samples, trace = tfm.sample_chain(
    num_results=1000,
    current_state=jnp.zeros(dim),
    kernel=adaptive,
    num_burnin_steps=1000,
    seed=key,
)
```

**BlackJAX:**
```python
import blackjax

# Warmup (builds an initial step size and mass matrix)
warmup = blackjax.window_adaptation(blackjax.nuts, log_prob)
(state, params), _ = warmup.run(key, jnp.zeros(dim), num_steps=1000)

# Single step (must be JIT-ted and scanned manually)
kernel = blackjax.nuts(log_prob, **params)

@jax.jit
def one_step(state, key):
    state, info = kernel.step(key, state)
    return state, (state.position, info)

keys = jax.random.split(key, 1000)
_, (samples, info) = jax.lax.scan(one_step, state, keys)
```

### Philosophical differences

| Concern | TFP | BlackJAX |
|---------|-----|---------|
| **State management** | Hidden inside `sample_chain` | Explicit; you own the state |
| **PRNG flow** | `sample_chain` splits the key internally | You split keys explicitly |
| **Custom step loop** | Override `trace_fn` or subclass kernel | Just rewrite the scan |
| **Mass matrix** | Via `SimpleStepSizeAdaptation` + custom kernel | `window_adaptation` returns it |
| **Multiple chains** | `jax.vmap` over `sample_chain` | `jax.vmap` over `one_step` |
| **Algorithm variety** | HMC, NUTS, RWM, Slice, REMC | HMC, NUTS, MALA, MCLMC, Pathfinder, SMC |
| **Production maturity** | High (Google-supported) | Growing rapidly |

### Multiple chains in TFP

```python
# vmap over initial states and seeds
n_chains = 4
init_states = jnp.zeros((n_chains, dim))
seeds = jax.random.split(key, n_chains)

def single_chain(init, seed):
    samples, _ = tfm.sample_chain(
        num_results=1000,
        current_state=init,
        kernel=nuts_kernel,
        seed=seed,
    )
    return samples

# samples shape: (n_chains, num_results, dim)
all_samples = jax.vmap(single_chain)(init_states, seeds)
```

R-hat and ESS across chains can be computed with
`tfp.mcmc.effective_sample_size` and `tfp.mcmc.potential_scale_reduction`.

In [ ]:
# ── 6.7  Multi-chain diagnostics with TFP utilities ──────────────────────────

print('Multi-chain sampling and R-hat diagnostics:')
print('='*60)

n_chains  = 4
n_warmup  = 500
n_samples_mc = 1000

@jax.jit
def single_chain(init_state, seed):
    kernel = tfm.NoUTurnSampler(
        target_log_prob_fn=banana_log_prob,
        step_size=0.5,
    )
    adaptive = tfm.DualAveragingStepSizeAdaptation(
        inner_kernel=kernel,
        num_adaptation_steps=int(0.8 * n_warmup),
    )
    samples, _ = tfm.sample_chain(
        num_results=n_samples_mc,
        current_state=init_state,
        kernel=adaptive,
        num_burnin_steps=n_warmup,
        seed=seed,
    )
    return samples

key, *chain_keys = jax.random.split(key, n_chains + 1)
chain_keys = jnp.array(chain_keys)

# Random initialisations for the chains
key, k_init = jax.random.split(key)
init_states = jax.random.normal(k_init, (n_chains, dim)) * 3.

# vmap over chains (parallel)
all_samples = jax.vmap(single_chain)(init_states, chain_keys)
print(f'  all_samples shape: {all_samples.shape}  # (chains, samples, dim)')
print()

# TFP diagnostics
# ESS: shape (dim,)
ess = tfm.effective_sample_size(all_samples)
print(f'  Effective sample size: {ess.round(1)}')

# R-hat (potential scale reduction factor)
# Values close to 1.0 indicate convergence
rhat = tfm.potential_scale_reduction(all_samples)
print(f'  R-hat: {rhat.round(4)}  (< 1.01 indicates convergence)')
print()

# Plot chains
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i in range(n_chains):
    axes[0].plot(all_samples[i, :200, 0], lw=0.7, alpha=0.7, label=f'chain {i}')
    axes[1].plot(all_samples[i, :200, 1], lw=0.7, alpha=0.7, label=f'chain {i}')
axes[0].set_title('x[0] traces (4 chains)')
axes[1].set_title('x[1] traces (4 chains)')
for ax in axes:
    ax.set_xlabel('iteration')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('multi_chain.png', dpi=100)
plt.show()